# From Script to Package: Structure, Errors, Config

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 4 §4.4–4.6 · type: walkthrough*  🔧 *This is the chapter's Build.*

**The promise:** by the end you can lay out a `src/` package, model expected-vs-unexpected failures with a typed exception hierarchy, log **structured** events, and load one validated `Settings` from the environment — the `core/` trio the capstone is built on.

Runs fully **offline and free**: everything is written to a throwaway temp directory and nothing is committed or networked. No API key, nothing billed.

## 🧠 Why this matters

There is a wide gap between Python that runs once on your machine and Python that runs unattended, at 3 a.m., under load, maintained by people who aren't you — and most of an agentic platform lives on the far side of that gap. Four habits close it: a **package layout** with one-way dependencies, an **error model** that separates expected failures from bugs, **structured logging** you can actually search, and **validated configuration** from the environment.

This notebook builds the chapter's 🔧 `core/` trio — `core/config.py`, `core/errors.py`, `core/logging.py`. It is "small, boring, and load-bearing": every later chapter (the FastAPI app, the agents, the workers) imports these three modules instead of reinventing config, errors, and logging. Getting them right once pays rent for the whole capstone. See §4.4–4.6.

## Objectives & prereqs

**By the end you can:**
- Read a `pyproject.toml` (declared ranges) vs a lockfile (pinned tree) and say why both are committed.
- Explain the `src` layout and the **one-way dependency rule** (`api → agents → tools → core`), and fix a **circular import** by moving a shared type down into `core/`.
- Build a `CapstoneError` / `ToolError` hierarchy and translate at a boundary with `raise ... from e`.
- Wire a **JSON log handler** so `extra={...}` fields actually appear — the pitfall that silently drops them under the default formatter.
- Load one validated `Settings` from the env that **fails fast** on a missing required var.

**Prereqs:** [`04-01`](04-01-mutability-typing-drills.ipynb) and [`04-02`](04-02-async-and-the-event-loop.ipynb).

**Packages:** standard library + **`pydantic`** (in `requirements.txt`). The `Settings` section uses **`pydantic-settings`** — *declare this extra* (`pip install pydantic-settings`); if it is absent the notebook falls back to a tiny stdlib shim so it still runs fully offline.

**Run first:** the Setup cell. Defaults to `MOCK=1` (here: fully offline — a throwaway package tree in a temp dir; nothing committed).

In [ ]:
# --- Setup -------------------------------------------------------------------
import importlib
import json
import logging
import os
import shutil
import sys
import tempfile
import textwrap
import traceback
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): everything happens in a temp directory and the process env,
# fully offline. There is no live path; the switch is kept for a uniform contract.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# A throwaway workspace for the package tree we build, torn down at the end.
WORK = Path(tempfile.mkdtemp(prefix="ch04_core_"))

print(f"MOCK mode: {MOCK}  (offline; building in a temp dir)")
print(f"workspace: {WORK}")
print(f"python: {sys.version.split()[0]}")

## 1 · `pyproject.toml` vs the lockfile

Project metadata and dependencies live in one standard file, `pyproject.toml`. Note the **two-level discipline**: your *declared* dependencies are loose ranges — what your code needs — while the **lockfile** (`uv.lock`, `poetry.lock`) pins the exact version and hash of every package in the resolved tree. Declared ranges express *intent*; the lockfile guarantees *reproducibility*. Commit both.

The cell writes a representative `pyproject.toml` so you can see the shape (we don't run a resolver here).

In [ ]:
pyproject = textwrap.dedent('''
    [project]
    name = "capstone"
    version = "0.1.0"
    requires-python = ">=3.12"
    dependencies = [
        "fastapi>=0.115",     # DECLARED: a loose range -- what the code needs
        "pydantic>=2.7",
        "httpx>=0.27",
    ]

    [dependency-groups]
    dev = ["pytest>=8", "ruff", "mypy"]
''').strip()

(WORK / "pyproject.toml").write_text(pyproject, encoding="utf-8")
print((WORK / "pyproject.toml").read_text(encoding="utf-8"))
print("\nThe lockfile (uv.lock / poetry.lock) would PIN every resolved package +")
print("hash so laptop, CI, and the prod container install byte-identical trees.")

As of early 2026 the tool to reach for is **uv** — a single fast binary managing Python versions, venvs, dependencies, and lockfiles. The workflow is three commands (explained here, not run live, since they touch the network):

- `uv sync` — create `.venv` and install exactly what `uv.lock` says.
- `uv add httpx` — add a dependency, re-resolve, and update the lockfile.
- `uv run pytest` — run a command inside the project environment.

The point of a lockfile: nothing changes until you run an *explicit* upgrade — so the 3 a.m. incident from an unpinned transitive dependency releasing a breaking change is a rite of passage you can simply skip.

## 2 · The `src` layout and the one-way dependency rule

A layout that scales from day one is the **src layout** — it makes it impossible to accidentally import your code from the working directory instead of the installed package. Dependencies point **one way**, toward `core/`:

```text
capstone-project/
  pyproject.toml
  src/capstone-project/
    __init__.py
    api/      FastAPI app and routes        (Ch 25)
    agents/   agent loops and orchestration (Parts IV-V)
    tools/    tool implementations          (Ch 12)
    core/     config, logging, errors       (this chapter)
```

The rule: **`api` imports `agents` imports `tools` imports `core`, never backwards.** Shared types live *down* in `core/` so everyone can import them without a cycle. Let's scaffold it.

In [ ]:
SRC = WORK / "src" / "capstone"
for sub in ["", "api", "agents", "tools", "core"]:
    (SRC / sub).mkdir(parents=True, exist_ok=True)
    (SRC / sub / "__init__.py").write_text("", encoding="utf-8")

# Make the throwaway package importable for the rest of the notebook.
sys.path.insert(0, str(WORK / "src"))

print("package tree:")
for p in sorted((WORK / "src").rglob("*.py")):
    print("  ", p.relative_to(WORK))

## 3 · 🔮 Predict: why a circular import fails *mid-execution*

The first time a module is imported, Python finds it on `sys.path`, **executes the whole file top to bottom**, and caches it in `sys.modules`. A **circular import** breaks this: module A, mid-execution, imports B, which imports the half-initialized A — and reads a name that doesn't exist *yet*.

We'll create the cycle deliberately: `tools/registry.py` imports a `ToolSpec` from `agents/planner.py`, and `agents/planner.py` imports from `tools/registry.py`.

**Predict before running:** does this fail at *import time* with a clear message, or only later when the function is called? And what *kind* of error?

In [ ]:
# agents/planner.py imports from tools/registry.py ...
(SRC / "agents" / "planner.py").write_text(textwrap.dedent('''
    from capstone.tools.registry import run_tool   # needs registry at import time

    class ToolSpec:                                 # ... and registry wants THIS
        name: str
'''), encoding="utf-8")

# ... while tools/registry.py imports ToolSpec back from agents/planner.py -> cycle
(SRC / "tools" / "registry.py").write_text(textwrap.dedent('''
    from capstone.agents.planner import ToolSpec    # circular: planner isn't ready

    def run_tool(spec: "ToolSpec") -> str:
        return f"running {spec.name}"
'''), encoding="utf-8")

for mod in list(sys.modules):
    if mod.startswith("capstone."):
        del sys.modules[mod]

try:
    importlib.import_module("capstone.agents.planner")
    print("imported with no error (unexpected)")
except ImportError as exc:
    print(f"{type(exc).__name__}: {exc}")
    print("\n^ Fails at IMPORT time: planner paused to import registry, which tried")
    print("  to read ToolSpec from the still-half-built planner. Classic cycle.")

**What you just saw.** An `ImportError` at *import time* — not a later runtime surprise. `planner` started executing, paused to import `registry`, which tried to read `ToolSpec` from the half-initialized `planner` before that line had run. The fix is **structural**: move the shared type *down* into `core/`, which both sides import, so the dependency arrow only ever points one way.

In [ ]:
# FIX: ToolSpec is a shared type -> it belongs in core/, which both sides import.
(SRC / "core" / "types.py").write_text(textwrap.dedent('''
    class ToolSpec:
        name: str
'''), encoding="utf-8")

(SRC / "agents" / "planner.py").write_text(textwrap.dedent('''
    from capstone.core.types import ToolSpec        # down into core -- no cycle
    from capstone.tools.registry import run_tool
'''), encoding="utf-8")

(SRC / "tools" / "registry.py").write_text(textwrap.dedent('''
    from capstone.core.types import ToolSpec        # down into core -- no cycle

    def run_tool(spec: "ToolSpec") -> str:
        return f"running {spec.name}"
'''), encoding="utf-8")

for mod in list(sys.modules):
    if mod.startswith("capstone."):
        del sys.modules[mod]

importlib.import_module("capstone.agents.planner")
print("imported cleanly: the shared type moved DOWN into core/, breaking the cycle.")
print("dependency arrow now points one way: agents -> tools -> core.")

**What you just saw.** Moving `ToolSpec` into `core/types.py` — a module both sides import — broke the cycle. This is the structural fix the chapter previews: shared types and interfaces live in `core/`, and dependencies point one way toward it. No clever import ordering, no local imports inside functions; just the right place for the type.

## 4 · 🔧 `core/errors.py`: a typed exception hierarchy

Production code distinguishes **expected** failures (a tool times out, a user exceeds quota) from **bugs** (a `KeyError` you never anticipated). Model the first kind explicitly with a small hierarchy, and let the second kind crash loudly so it gets fixed. At a boundary, **translate** an upstream error into your own and preserve the cause with `raise ... from e`.

In [ ]:
errors_py = textwrap.dedent('''
    class CapstoneError(Exception):
        # Base for all EXPECTED application errors.
        pass

    class ToolError(CapstoneError):
        def __init__(self, tool: str, message: str):
            self.tool = tool
            super().__init__(f"{tool}: {message}")
''')
(SRC / "core" / "errors.py").write_text(errors_py, encoding="utf-8")

for mod in list(sys.modules):
    if mod.startswith("capstone.core.errors"):
        del sys.modules[mod]
errors = importlib.import_module("capstone.core.errors")
CapstoneError, ToolError = errors.CapstoneError, errors.ToolError


# Simulate a boundary: an upstream failure translated into OUR error, cause kept.
class HTTPError(Exception):
    pass


def call_upstream_tool():
    raise HTTPError("connection reset by peer")   # the messy upstream cause


try:
    try:
        call_upstream_tool()
    except HTTPError as e:
        raise ToolError("web_fetch", "upstream request failed") from e   # <- from e
except ToolError as exc:
    print("caught:", repr(exc))
    print("is a CapstoneError?", isinstance(exc, CapstoneError))
    print("\n--- chained traceback (the root cause SURVIVES via `from e`) ---")
    print("".join(traceback.format_exception(type(exc), exc, exc.__traceback__))[-600:])

**What you just saw.** `raise ToolError(...) from e` chained the original `HTTPError` to our typed `ToolError` — the traceback shows *"The above exception was the direct cause..."*, so the root cause is never lost. `ToolError` is a `CapstoneError`, so a single `except CapstoneError` at the top of a request handler catches every *expected* failure while real bugs still crash loudly.

The anti-pattern to hunt down — especially in AI-generated code — is the **silent swallow** `except Exception: pass`. It converts a clear failure today into an unexplainable wrong answer next month.

## 5 · ⚠️ Pitfall: `extra={...}` log fields are silently dropped

Log **events with fields** ("what happened, with which IDs"), not prose. But here is the caveat that bites everyone once: the stdlib's **default text formatter ignores any `extra` key it has no `%(...)s` placeholder for.** Under a plain `logging.basicConfig()`, your `tool`, `thread_id`, and `ms` are *silently dropped* and the line logs as bare prose.

**🔮 Predict:** in the cell below we log with `extra={"tool": ..., "thread_id": ..., "ms": ...}` under the *default* formatter. Will those three fields appear in the output line, or vanish?

In [ ]:
import io

# A logger writing to an in-memory buffer with the DEFAULT text formatter.
buf_default = io.StringIO()
logger = logging.getLogger("capstone.demo.default")
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.propagate = False
h = logging.StreamHandler(buf_default)
h.setFormatter(logging.Formatter("%(levelname)s %(message)s"))  # no placeholders for extras
logger.addHandler(h)

logger.info("tool_call_finished", extra={"tool": "web_fetch", "thread_id": "t-42", "ms": 318})

print("DEFAULT formatter output:")
print(" ", buf_default.getvalue().strip())
print("\n-> the tool / thread_id / ms you attached are NOWHERE in the line. Dropped.")

**The fix:** wire a **structured (JSON) handler** that reads the `extra` fields off the log record and emits them. Now the same `logger.info(..., extra={...})` call produces a searchable, aggregatable event — exactly what Chapter 23 turns into telemetry.

In [ ]:
# A minimal JSON formatter: pull our known fields off the LogRecord if present.
_STD = set(vars(logging.makeLogRecord({})))   # the record's built-in attribute names


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload = {"level": record.levelname, "event": record.getMessage()}
        # Anything not on a vanilla record was passed via extra={...}: include it.
        for key, value in record.__dict__.items():
            if key not in _STD and not key.startswith("_"):
                payload[key] = value
        return json.dumps(payload)


buf_json = io.StringIO()
jlogger = logging.getLogger("capstone.demo.json")
jlogger.handlers.clear()
jlogger.setLevel(logging.INFO)
jlogger.propagate = False
jh = logging.StreamHandler(buf_json)
jh.setFormatter(JsonFormatter())
jlogger.addHandler(jh)

jlogger.info("tool_call_finished", extra={"tool": "web_fetch", "thread_id": "t-42", "ms": 318})

line = buf_json.getvalue().strip()
print("JSON formatter output:")
print(" ", line)
parsed = json.loads(line)
print("\nnow searchable -> parsed['tool'] =", parsed["tool"], "| ms =", parsed["ms"])

**What you just saw.** Same `logger.info(..., extra={...})` call, two very different outcomes. Under the default text formatter the structured fields vanished; under the JSON handler they became real, queryable keys. Configure structured logging **once** at startup (`core/logging.py`) and every module's `logger.info("event", extra={...})` just works. This is the whole reason the book logs events-with-fields instead of f-string prose.

## 6 · 🔧 `core/config.py`: one validated `Settings`, fail-fast on a missing var

Configuration follows one rule: **code is the same everywhere; config differs per environment, and secrets never live in either the code or the repo.** Read config from the environment into one typed, validated object at startup. `pydantic-settings` makes this a few lines and **fails fast** on a missing or malformed value.

> If `pydantic-settings` is not installed, the cell uses a tiny stdlib shim with the same behaviour so the notebook still runs offline. To use the real thing: `pip install pydantic-settings`.

In [ ]:
try:
    from pydantic_settings import BaseSettings, SettingsConfigDict
    BACKEND = "pydantic-settings"
except ImportError:
    # Minimal offline shim: same shape (env_prefix, required vs default, fail-fast).
    BACKEND = "stdlib-shim"

    class SettingsConfigDict(dict):
        pass

    class BaseSettings:
        model_config = {"env_prefix": ""}

        def __init__(self, **overrides):
            prefix = self.model_config.get("env_prefix", "")
            ann = {}
            for klass in reversed(type(self).__mro__):
                ann.update(getattr(klass, "__annotations__", {}))
            for field, ftype in ann.items():
                if field in overrides:
                    raw = overrides[field]
                elif (prefix + field.upper()) in os.environ:
                    raw = os.environ[prefix + field.upper()]
                elif hasattr(type(self), field):
                    raw = getattr(type(self), field)        # class-level default
                else:
                    raise ValueError(
                        f"missing required setting: {prefix + field.upper()}"
                    )
                setattr(self, field, ftype(raw) if ftype in (int, float, str) else raw)

print("Settings backend:", BACKEND)

In [ ]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="CAPSTONE_")

    env: str = "dev"
    database_url: str          # REQUIRED -- startup fails if missing
    llm_api_key: str           # injected via env/secret manager, never committed
    request_timeout: float = 30.0


# Inject config via the environment (never hardcoded). Secrets come from env only.
os.environ["CAPSTONE_DATABASE_URL"] = "postgresql://localhost/capstone"
os.environ["CAPSTONE_LLM_API_KEY"] = "sk-FAKE-only-from-env-never-committed"

settings = Settings()
print("loaded settings:")
print("  env             =", settings.env, "(default)")
print("  database_url     =", settings.database_url)
print("  request_timeout  =", settings.request_timeout, "(default)")
print("  llm_api_key      =", settings.llm_api_key[:7] + "..." )   # never log full secrets

### 🔮 Predict: a missing required var

`database_url` has no default. Predict: if we *unset* `CAPSTONE_DATABASE_URL` and construct `Settings()` again, does it (a) fall back to some empty value, or (b) raise immediately at startup?

In [ ]:
saved = os.environ.pop("CAPSTONE_DATABASE_URL")   # simulate a missing required var
try:
    Settings()
    print("constructed with no error (unexpected)")
except Exception as exc:
    print(f"FAILED FAST: {type(exc).__name__}")
    print("  ->", str(exc).splitlines()[0])
    print("\nThis is the point: a misconfigured deploy dies at STARTUP with a clear")
    print("message, not 200ms into the first request with a confusing one.")
finally:
    os.environ["CAPSTONE_DATABASE_URL"] = saved    # restore for later cells

**What you just saw.** A missing required var made `Settings()` raise **immediately** — fail-fast at startup, with a message naming the field. That is exactly what you want: a bad deploy crashes on boot in CI or the container's first second, not as a mysterious wrong answer under load. And notice the secret only ever came from the environment — never a literal in the code or the repo.

## 📋 Production-Python self-audit

The chapter's checklist, as a cell you can run against any module you write:

In [ ]:
checklist = {
    "Mutability: no mutable default args; closures in loops bind values": True,
    "Typing: public funcs annotated; mypy/pyright in CI; vendors as Protocols": True,
    "Async: I/O paths async end-to-end; no blocking in coroutines; bounded fan-out": True,
    "Packaging: pyproject.toml + committed lockfile; one venv per project": True,
    "Structure: src layout; absolute imports; deps point one way toward core/": True,
    "Errors: typed hierarchy; raise ... from e at boundaries; no bare except: pass": True,
    "Logging/config: structured events with IDs; one validated Settings from env": True,
}
width = max(len(k) for k in checklist)
for item, ok in checklist.items():
    print(f"[{'x' if ok else ' '}] {item:<{width}}")
print("\nThis notebook demonstrated every row -- the core/ trio is the load-bearing part.")

In [ ]:
# Tidy up the throwaway workspace (nothing here was meant to persist).
shutil.rmtree(WORK, ignore_errors=True)
if str(WORK / "src") in sys.path:
    sys.path.remove(str(WORK / "src"))
print("cleaned up:", WORK)

## 🎯 Senior lens

`core/` should be **"small, boring, and load-bearing."** A senior resists the urge to make it clever: it is config, errors, and logging — the three things *every* other module needs and none should reimplement. The payoff is leverage. Because the FastAPI app, the agents, and the workers all `from capstone.core...` import the same `Settings`, the same `CapstoneError` base, and the same structured logger, a change to how the system is configured or how it logs is a one-file change, not a sweep across the codebase.

The deepest of these is the **dependency direction**. "Everything points down toward `core/`, nothing points back up" is what keeps a growing platform from collapsing into a ball of cycles. When you feel the pull to import `agents` from `core` to reuse one type, that is the signal the type belongs *in* `core/`. Get the arrows right early; they are expensive to reverse once a hundred modules depend on them.

## Recap

- **`pyproject.toml` declares loose ranges; the lockfile pins the resolved tree** — commit both for reproducible installs; `uv sync` / `uv add` / `uv run` is the modern workflow.
- The **src layout** + a **one-way dependency rule** (`api → agents → tools → core`) prevents accidental local imports and import cycles.
- A **circular import fails at import time**; the fix is structural — move the shared type *down* into `core/`.
- A **typed error hierarchy** (`CapstoneError` / `ToolError`) separates expected failures from bugs; `raise ... from e` preserves the cause; never `except: pass`.
- `extra={...}` log fields are **silently dropped** under the default formatter — wire a **JSON handler** so they become searchable events.
- One validated **`Settings`** from the environment **fails fast** on a missing var; secrets come from env only, never the code or repo.

## Exercises

Each exercise *changes* something and asks you to predict the result first. (Solutions land in `solutions/` in Phase 2.)

1. **Add `RateLimitError`.** Extend the hierarchy with `RateLimitError(CapstoneError)` carrying a `retry_after: float`. Predict whether a top-level `except CapstoneError` still catches it (and why), then confirm. What does that say about where to put the `except`?
2. **Validate a config range.** Give `Settings` a `max_concurrency: int` field constrained to `1..100` (with `pydantic`, use `Field(ge=1, le=100)`). Set `CAPSTONE_MAX_CONCURRENCY=999`, predict the exact failure, and confirm it dies at startup, not at use.
3. **Break the formatter on purpose.** Add a `%(thread_id)s` placeholder to the *default* text formatter but log *without* that key in `extra`. Predict what happens (hint: it's not silent this time), then fix it. Why is a JSON handler more robust than placeholders?
4. **Re-introduce the cycle.** Make `core/types.py` import from `agents/planner.py`. Predict which direction now violates the one-way rule, run it to see the cycle return, and explain why `core/` must never import upward.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

In [ ]:
# Exercise 4 -- your code here

## Next

You built the `core/` trio in a temp dir; in the capstone it is real, permanent, and imported everywhere.

- ▶️ **Next notebook:** [`../05-clean-code-and-design/PLAN.md`](../05-clean-code-and-design/) — Ch 5 grows the provider/domain *seams* on top of the `Protocol` idea from 04-01 and the `core/` foundation here.
- 📖 **Book:** revisit §4.4 (packaging/uv), §4.5 (structure & imports), and §4.6 (errors, logging, configuration — the 🔧 Build).
- 🎓 **Capstone:** this *is* `capstone-project/.../core/config.py`, `core/errors.py`, and `core/logging.py` — the chapter's 🔧 Build. Every later part imports the `Settings`, the `CapstoneError` hierarchy, and the structured logger you built here instead of reinventing them.